In [2]:
#Human in Loop

In [7]:
from typing_extensions import TypedDict,Literal
from langgraph.graph import StateGraph ,END,START

In [8]:
### STATE
class ApprovalState(TypedDict):
    request : str
    generated_response : str
    human_decision : str
    final_status : str

In [10]:
### NODE 1: AI generates response
def generate_response(state: ApprovalState) -> dict[str,str]:
    return{
        "generated_response" : f"AI generated response for request: {state['request']}"
    }

### NODE 2: Human Approval Node
def human_approval(state: ApprovalState) -> dict[str,str]:
    print("\n Generated response:")
    print(state['generated_response'])
    decision = input("Do you approve the response? (yes/no): ")
    return{
        "human_decision" : decision.lower()
    }

### NODE 3 : Approved
def approved(state: ApprovalState) -> dict[str,str]:
    return{
        "final_status" : "Request Approved and sent"
    }

### NODE 4 : Rejected
def rejected(state: ApprovalState) -> dict[str,str]:
    return{
        "final_status" : "Request Rejected"
    }

### conditional routing
def route_decision(state: ApprovalState) -> Literal['approved'] | Literal['rejected']:
    if state['human_decision'] == 'yes':
        return "approved"
    else:
        return "rejected"


### Build Graph

builder = StateGraph(ApprovalState)

builder.add_node("generate_response", generate_response)
builder.add_node("human_approval", human_approval)
builder.add_node("approved", approved)
builder.add_node("rejected", rejected)

builder.add_edge(START, "generate_response")
builder.add_edge("generate_response", "human_approval")

builder.add_conditional_edges(
    "human_approval", route_decision, {
        "approved": "approved",
        "rejected": "rejected" 
    }
)

builder.add_edge("approved", END)
builder.add_edge("rejected", END)

graph = builder.compile()
result = graph.invoke({
    "request":"Leave request for 2 days"
})

print("\nFinal Result:")
print(result)


 Generated response:
AI generated response for request: Leave request for 2 days

Final Result:
{'request': 'Leave request for 2 days', 'generated_response': 'AI generated response for request: Leave request for 2 days', 'human_decision': 'no', 'final_status': 'Request Rejected'}
